# Data Wrangling and Exploratory Data Analysis

This notebook uses IBM's course-prepared launch data for reproducibility. The `Class` target is 1 for a successful first-stage landing and 0 otherwise. Rates should be interpreted alongside sample counts.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.common import load_course_csv, ensure_class

In [ ]:
df = ensure_class(load_course_csv('dataset_part_2.csv'))
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
for col in ['PayloadMass','FlightNumber','Class']:
    if col in df: df[col] = pd.to_numeric(df[col], errors='coerce')
df['LaunchYear'] = df['Date'].dt.year
print('Shape:', df.shape)
display(df.head())
display((df.isna().mean()*100).sort_values(ascending=False).head(10).rename('missing_pct').to_frame())

In [ ]:
# Overall landing class balance
class_counts = df['Class'].value_counts().sort_index()
ax = class_counts.plot(kind='bar', color=['#e76f51','#12a89d'], title='Landing class counts')
ax.set_xlabel('Class (0 = unsuccessful, 1 = successful)'); ax.set_ylabel('Launch records'); plt.tight_layout(); plt.show()
print('Overall landing success rate:', round(df['Class'].mean()*100, 1), '%')

In [ ]:
# Launch-site counts and success rates
site = df.groupby('LaunchSite').agg(launches=('Class','size'), success_rate=('Class','mean')).sort_values('launches', ascending=False)
display(site.assign(success_rate_pct=100*site['success_rate']))
site['success_rate'].mul(100).plot(kind='bar', title='Landing success rate by launch site', ylabel='Success rate (%)', ylim=(0,100), color='#12a89d')
plt.xticks(rotation=20, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
# Success rates by orbit and year
orbit = df.groupby('Orbit').agg(launches=('Class','size'), success_rate=('Class','mean')).sort_values('success_rate', ascending=False)
display(orbit.assign(success_rate_pct=100*orbit['success_rate']))
yearly = df.dropna(subset=['LaunchYear']).groupby('LaunchYear').agg(launches=('Class','size'), success_rate=('Class','mean'))
display(yearly.assign(success_rate_pct=100*yearly['success_rate']))
yearly['success_rate'].mul(100).plot(marker='o', title='Landing success rate by year', ylabel='Success rate (%)', ylim=(0,100), color='#12a89d')
plt.grid(alpha=.25); plt.tight_layout(); plt.show()

In [ ]:
# Payload mass vs flight experience: relationship is exploratory, not causal
fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(df['FlightNumber'], df['PayloadMass'], c=df['Class'], alpha=.8)
ax.set(title='Payload mass vs flight number', xlabel='Flight number', ylabel='Payload mass (kg)')
plt.tight_layout(); plt.show()